In [60]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [61]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [62]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "python" or "json" or "regex"
        "solution_criteria": "Describe a little bit better some points that are crucial to the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text=chat(messages, stop_sequences=["```"])
    return json.loads(text)
    

In [63]:
dataset = generate_dataset()
dataset

[{'task': "Parse an AWS CloudWatch log entry and extract the timestamp, log level, and message. The log format is '[TIMESTAMP] [LEVEL] MESSAGE'",
  'format': 'regex',
  'solution_criteria': 'The regex should correctly capture three groups: ISO 8601 timestamp format, log level (ERROR, WARN, INFO, DEBUG), and the remaining message text. It should handle variable-length messages and different timestamp formats.'},
 {'task': 'Write a Python function that takes an AWS S3 bucket name and returns True if it follows AWS naming conventions (lowercase, 3-63 characters, no consecutive hyphens, starts/ends with alphanumeric)',
  'format': 'python',
  'solution_criteria': 'The function should validate all S3 bucket naming rules including length constraints, character restrictions, hyphen placement, and return a boolean. It should handle edge cases like empty strings and special characters gracefully.'},
 {'task': 'Create a JSON configuration object for an AWS Lambda function that includes environme

In [64]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [65]:
def run_prompt(test_case):
    """Merge the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:
{test_case['task']}
* Respond only with Python, Json or plain regex
*Do not add any explanations or comments
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [ ]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.
    
    Original Task:
    <task>
    {test_case["task"]}
    </task>
    
    Solution to Evaluate:
    <solution>
    {output}
    </solution>

    solution criteria:
    <criteria>
    {test_case["solution_criteria"]}
    </criteria>
    
    Output Format
    Provide your evaluation as a structured JSON object with the following fields, in this specific order:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your overall assessment
    - "solution_criteria_compliance": A short description of how well the solution meets the criteria outlined in the task
    - "score": A number between 1-10
    
    Respond with JSON. Keep your response concise and direct.
    Example response shape:
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number
    }}
        """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [67]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [68]:
def run_test_case(test_case):
    """Calls run_promopt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    sintax_score = grade_syntax(output, test_case)
    score = (model_score + sintax_score) / 2
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [69]:
from statistics import mean
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results =[]
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average Score: {average_score:.2f}")
    return results

In [70]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

result = run_eval(dataset)

Average Score: 7.00


In [71]:
print(json.dumps(result, indent=2))

[
  {
    "output": "\nimport re\nimport json\n\ndef parse_cloudwatch_log(log_entry):\n    pattern = r'\\[(.+?)\\]\\s+\\[(.+?)\\]\\s+(.*)'\n    match = re.match(pattern, log_entry)\n    \n    if match:\n        return {\n            \"timestamp\": match.group(1),\n            \"level\": match.group(2),\n            \"message\": match.group(3)\n        }\n    return None\n\nlog_entry = \"[2023-10-15T10:30:45Z] [ERROR] Database connection failed\"\nresult = parse_cloudwatch_log(log_entry)\nprint(json.dumps(result, indent=2))\n",
    "test_case": {
      "task": "Parse an AWS CloudWatch log entry and extract the timestamp, log level, and message. The log format is '[TIMESTAMP] [LEVEL] MESSAGE'",
      "format": "regex",
      "solution_criteria": "The regex should correctly capture three groups: ISO 8601 timestamp format, log level (ERROR, WARN, INFO, DEBUG), and the remaining message text. It should handle variable-length messages and different timestamp formats."
    },
    "score": 3.5